In [1]:
# Load dataset

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load WNBA dataset
df_master = pd.read_csv('wnba_master_dataset.csv', parse_dates=['GAME_DATE'])

# Convert string columns
df_master['HOME_AWAY'] = df_master['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})
df_master['POSITION']  = df_master['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

print(f"Rows:    {len(df_master):,}")
print(f"Columns: {len(df_master.columns)}")
print(f"Seasons: {sorted(df_master['SEASON'].unique())}")
print(f"Date range: {df_master['GAME_DATE'].min().date()} to {df_master['GAME_DATE'].max().date()}")

Rows:    15,882
Columns: 97
Seasons: [2021, 2022, 2023, 2024, 2025, 2026]
Date range: 2021-05-21 to 2026-06-23


In [5]:
# Feature list with updated H2H features

target_cols = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

feature_cols = [
    # 5-game rolling averages
    'PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
    'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
    'FGA_roll5', 'FG3A_roll5',

    # 10-game rolling averages
    'PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
    'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
    'FGA_roll10', 'FG3A_roll10',

    # Situational
    'HOME_AWAY', 'DAYS_REST', 'DEF_RATING', 'PACE',

    # Opponent stats
    'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG', 'OPP_AST_ALLOWED_PG',
    'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG',

    # Usage and position
    'USG_PCT',
    'OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
    'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS',

    # Rolling features
    'USG_PCT_roll5',
    'OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm',
    'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm',
    'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm',

    # Team usage context
    'RELATIVE_USG', 'USG_RANK',

    # Volatility
    'PTS_std_roll10', 'REB_std_roll10',
    'PTS_cv_roll10', 'REB_cv_roll10',

    # WNBA specific
    'IS_ROOKIE_SEASON',
    'GAMES_PLAYED',

    # NEW — Head-to-head history features
    'H2H_PTS_AVG', 'H2H_REB_AVG', 'H2H_AST_AVG',
    'H2H_FG3M_AVG', 'H2H_GAMES',
]

print(f"Total features: {len(feature_cols)}")

Total features: 57


In [7]:
# Recompute position-normalized features and refit scaler

from sklearn.preprocessing import StandardScaler

pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]

for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master[norm_col] = df_master.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

df_raw = df_master.dropna(subset=feature_cols).reset_index(drop=True)

scaler = StandardScaler()
scaler.fit(df_raw[feature_cols])

test       = df_raw[feature_cols].iloc[0:1].values
normalized = scaler.transform(test)

print(f"Rows after NaN drop: {len(df_raw):,}")
print(f"Scaler fit on {scaler.n_features_in_} features")

with open('wnba_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"✅ WNBA scaler saved")

Rows after NaN drop: 14,479
Scaler fit on 57 features
✅ WNBA scaler saved


In [9]:
# Train/Val split

df_master_clean = df_master.dropna(subset=feature_cols).reset_index(drop=True)

# Dynamic split — validate on the most recent 14 days of 2026 data
# Train on everything before that (2021-2025 full history + early/mid 2026)
latest_date = df_master_clean['GAME_DATE'].max()
SPLIT_DATE  = latest_date - pd.Timedelta(days=14)

print(f"Latest game date: {latest_date.date()}")
print(f"Split date:       {SPLIT_DATE.date()}")
print()

df_train = df_master_clean[
    df_master_clean['GAME_DATE'] < SPLIT_DATE
].copy().reset_index(drop=True)

df_val = df_master_clean[
    df_master_clean['GAME_DATE'] >= SPLIT_DATE
].copy().reset_index(drop=True)

print(f"Training rows:   {len(df_train):,}")
print(f"Validation rows: {len(df_val):,}")
print()
print(f"Training 2026 rows:   {(df_train['SEASON']==2026).sum():,}")
print(f"Validation 2026 rows: {(df_val['SEASON']==2026).sum():,}")
print()

# Normalize and convert to tensors
X_train_norm = scaler.transform(df_train[feature_cols].values.astype(np.float32))
X_val_norm   = scaler.transform(df_val[feature_cols].values.astype(np.float32))

X_train = torch.tensor(X_train_norm, dtype=torch.float32)
y_train = torch.tensor(df_train[target_cols].values, dtype=torch.float32)

X_val   = torch.tensor(X_val_norm, dtype=torch.float32)
y_val   = torch.tensor(df_val[target_cols].values, dtype=torch.float32)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"NaNs in X_train: {torch.isnan(X_train).sum().item()}")

Latest game date: 2026-06-23
Split date:       2026-06-09

Training rows:   13,959
Validation rows: 520

Training 2026 rows:   533
Validation 2026 rows: 520

X_train shape: torch.Size([13959, 57])
X_val shape:   torch.Size([520, 57])
NaNs in X_train: 0


In [11]:
# Model architecture and initialization
# Fresh Xavier initialization with same architecture as NBA model
# (input_dim updated to 57 to include new H2H features)

torch.manual_seed(123)
np.random.seed(123)

class PlayerPropModel(nn.Module):
    def __init__(self, input_dim, target_stats):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.BatchNorm1d(128), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(),
            nn.BatchNorm1d(64), nn.Dropout(0.4),
        )
        self.heads = nn.ModuleDict({
            stat: nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2)
            ) for stat in target_stats
        })

    def forward(self, x):
        shared = self.trunk(x)
        outputs = {}
        for stat, head in self.heads.items():
            raw       = head(shared)
            mu        = raw[:, 0]
            log_sigma = torch.clamp(raw[:, 1], min=-3, max=3)
            sigma     = torch.exp(log_sigma) + 1e-6
            outputs[stat] = (mu, sigma)
        return outputs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

wnba_model = PlayerPropModel(
    input_dim=len(feature_cols),  # 57 now
    target_stats=target_cols
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

wnba_model.apply(init_weights)

print(f"WNBA model initialized with Xavier weights")
print(f"Input features: {len(feature_cols)}")
print()

total_params = sum(p.numel() for p in wnba_model.parameters())
print(f"Total parameters: {total_params:,}")

wnba_model = wnba_model.to(device)

WNBA model initialized with Xavier weights
Input features: 57

Total parameters: 28,940


In [13]:
# Training Loop

from torch.utils.data import TensorDataset, DataLoader

EPOCHS        = 40
BATCH_SIZE    = 128
LEARNING_RATE = 5e-4

def nll_loss(mu, sigma, target):
    distribution = torch.distributions.Normal(mu, sigma)
    return -distribution.log_prob(target).mean()

train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset   = TensorDataset(X_val, y_val)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.Adam(
    wnba_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=10
)

print(f"Training on: {device}")

best_val_loss = float('inf')
train_losses  = []
val_losses    = []

for epoch in range(EPOCHS):
    wnba_model.train()
    epoch_train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = wnba_model(X_batch)

        loss = sum(
            nll_loss(outputs[stat][0], outputs[stat][1], y_batch[:, i])
            for i, stat in enumerate(target_cols)
        ) / len(target_cols)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(wnba_model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)

    wnba_model.eval()
    epoch_val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = wnba_model(X_batch)
            loss = sum(
                nll_loss(outputs[stat][0], outputs[stat][1], y_batch[:, i])
                for i, stat in enumerate(target_cols)
            ) / len(target_cols)
            epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(wnba_model.state_dict(), 'wnba_best_model.pth')

    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.6f} | "
              f"{'✅ Best' if avg_val_loss == best_val_loss else ''}")

print()
print(f"Training complete — best val loss: {best_val_loss:.4f}")
print(f"Compare to previous best: 1.8967")

Training on: cpu
Epoch   5/40 | Train Loss: 2.1568 | Val Loss: 2.0644 | LR: 0.000500 | ✅ Best
Epoch  10/40 | Train Loss: 1.9752 | Val Loss: 1.9519 | LR: 0.000500 | ✅ Best
Epoch  15/40 | Train Loss: 1.9277 | Val Loss: 1.9252 | LR: 0.000500 | ✅ Best
Epoch  20/40 | Train Loss: 1.9099 | Val Loss: 1.9141 | LR: 0.000500 | 
Epoch  25/40 | Train Loss: 1.8879 | Val Loss: 1.8962 | LR: 0.000500 | ✅ Best
Epoch  30/40 | Train Loss: 1.8782 | Val Loss: 1.8870 | LR: 0.000500 | 
Epoch  35/40 | Train Loss: 1.8788 | Val Loss: 1.8847 | LR: 0.000500 | 
Epoch  40/40 | Train Loss: 1.8681 | Val Loss: 1.8788 | LR: 0.000500 | ✅ Best

Training complete — best val loss: 1.8788
Compare to previous best: 1.8967


In [15]:
import pickle

# Reload best model weights
wnba_model.load_state_dict(torch.load('wnba_best_model.pth', map_location=device))
wnba_model.eval()
print("✅ Best model weights reloaded")
print()

with open('wnba_scaler.pkl', 'rb') as f:
    wnba_scaler = pickle.load(f)
print("✅ Scaler loaded")
print()

# Check predictions on A'ja Wilson's most recent validation-set games
aja_val = df_master[
    (df_master['PLAYER_NAME'] == "A'ja Wilson") &
    (df_master['GAME_DATE'] >= SPLIT_DATE)
].dropna(subset=feature_cols).copy()

print(f"A'ja Wilson games in validation window: {len(aja_val)}")
print()

if len(aja_val) > 0:
    features_raw  = aja_val[feature_cols].values.astype(np.float32)
    features_norm = wnba_scaler.transform(features_raw)
    features_t    = torch.tensor(features_norm, dtype=torch.float32).to(device)

    with torch.no_grad():
        preds = wnba_model(features_t)

    print("A'ja Wilson — Predicted vs Actual Points:")
    print(f"{'Game':<12} {'Opp':<5} {'Actual':>8} {'Pred μ':>8} {'Pred σ':>8} {'Error':>8}")
    print("-" * 55)
    for i in range(len(aja_val)):
        actual = aja_val['PTS'].iloc[i]
        opp    = aja_val['OPPONENT'].iloc[i]
        mu     = preds['PTS'][0][i].item()
        sigma  = preds['PTS'][1][i].item()
        error  = abs(actual - mu)
        date   = str(aja_val['GAME_DATE'].iloc[i].date())
        print(f"{date:<12} {opp:<5} {actual:>8.1f} {mu:>8.1f} {sigma:>8.1f} {error:>8.1f}")

✅ Best model weights reloaded

✅ Scaler loaded

A'ja Wilson games in validation window: 6

A'ja Wilson — Predicted vs Actual Points:
Game         Opp     Actual   Pred μ   Pred σ    Error
-------------------------------------------------------
2026-06-11   PDX       32.0     24.4      7.4      7.6
2026-06-13   MIN       24.0     19.0      6.3      5.0
2026-06-15   DAL       18.0     23.0      7.1      5.0
2026-06-17   PHX       33.0     20.7      6.7     12.3
2026-06-21   GSV       19.0     23.3      7.0      4.3
2026-06-23   NYL       16.0     20.2      6.3      4.2


In [17]:
# Re-run full systematic bias check with retrained model
# Need to regenerate predictions through the live pipeline structure first
# For now, check average predicted vs actual PTS across all validation players

val_preds = []

for player in df_val['PLAYER_NAME'].unique():
    player_val = df_val[df_val['PLAYER_NAME'] == player]
    if len(player_val) == 0:
        continue

    features_raw  = player_val[feature_cols].values.astype(np.float32)
    features_norm = wnba_scaler.transform(features_raw)
    features_t    = torch.tensor(features_norm, dtype=torch.float32).to(device)

    with torch.no_grad():
        preds = wnba_model(features_t)

    for i in range(len(player_val)):
        val_preds.append({
            'player': player,
            'stat':   'PTS',
            'actual': player_val['PTS'].iloc[i],
            'pred':   preds['PTS'][0][i].item(),
        })

df_val_preds = pd.DataFrame(val_preds)

avg_actual = df_val_preds['actual'].mean()
avg_pred   = df_val_preds['pred'].mean()
avg_error  = (df_val_preds['actual'] - df_val_preds['pred']).mean()

print(f"Validation set ({len(df_val_preds)} player-games):")
print(f"  Avg actual PTS:    {avg_actual:.2f}")
print(f"  Avg predicted PTS: {avg_pred:.2f}")
print(f"  Avg signed error:  {avg_error:+.2f}  (positive = underpredicting)")
print(f"  Mean abs error:    {(df_val_preds['actual'] - df_val_preds['pred']).abs().mean():.2f}")

Validation set (520 player-games):
  Avg actual PTS:    12.34
  Avg predicted PTS: 11.08
  Avg signed error:  +1.26  (positive = underpredicting)
  Mean abs error:    4.60


In [19]:
val_preds_all = []

for player in df_val['PLAYER_NAME'].unique():
    player_val = df_val[df_val['PLAYER_NAME'] == player]
    if len(player_val) == 0:
        continue

    features_raw  = player_val[feature_cols].values.astype(np.float32)
    features_norm = wnba_scaler.transform(features_raw)
    features_t    = torch.tensor(features_norm, dtype=torch.float32).to(device)

    with torch.no_grad():
        preds = wnba_model(features_t)

    for i in range(len(player_val)):
        for stat in ['PTS', 'REB', 'AST', 'FG3M']:
            val_preds_all.append({
                'player': player,
                'stat':   stat,
                'actual': player_val[stat].iloc[i],
                'pred':   preds[stat][0][i].item(),
            })

df_val_all = pd.DataFrame(val_preds_all)

print("Bias check by stat (retrained model):")
print(f"{'Stat':<6} {'Avg Actual':>10} {'Avg Pred':>10} {'Signed Err':>11} {'Abs Err':>9}")
print("-" * 50)

for stat in ['PTS', 'REB', 'AST', 'FG3M']:
    stat_df    = df_val_all[df_val_all['stat'] == stat]
    avg_actual = stat_df['actual'].mean()
    avg_pred   = stat_df['pred'].mean()
    signed_err = avg_actual - avg_pred
    abs_err    = (stat_df['actual'] - stat_df['pred']).abs().mean()
    print(f"{stat:<6} {avg_actual:>10.2f} {avg_pred:>10.2f} {signed_err:>+11.2f} {abs_err:>9.2f}")

Bias check by stat (retrained model):
Stat   Avg Actual   Avg Pred  Signed Err   Abs Err
--------------------------------------------------
PTS         12.34      11.08       +1.26      4.60
REB          4.38       4.21       +0.16      1.77
AST          2.87       2.58       +0.29      1.47
FG3M         1.24       1.16       +0.08      0.87
